# Fetching the data

The data that is used in the paper is related to various financial instruments:

- GOLD --> Gold futures
- Silver --> Silver futures
- Wheat -->	Wheat futures
- GAZ -->	Natural gas futures
- SP500 -->	S&P 500	
- NSDQ -->	Nasdaq Composite
- VIX -->	Volatility index
- IBEX -->	IBEX 35 (Spain)
- AEX -->	AEX (Netherlands)
- BEL -->	BEL 20 (Belgium)
- DAX -->	DAX (Germany)
- FTSE -->	FTSE 100 (UK)
- SMI -->	Swiss Market Index
- ASX -->	ASX 200 (Australia)
- N225 -->	Nikkei 225
- HIS -->	Hang Seng
- Nifty -->	Nifty 50 (India)
- BVSP -->	Bovespa (Brazil)	
- NZX -->	NZX 50 (New Zealand)
- US yield  -->	US 10-year Treasury

In [ ]:
%pip install pandas requests yfinance

In [ ]:
import io
from pathlib import Path
 
import pandas as pd
import matplotlib.pyplot as plt
import requests
import yfinance as yf

In [ ]:
START = "2011-02-01"
END = "2026-02-28" 
PERIOD = '1d'

OUT = Path("data/raw")
OUT.mkdir(parents=True, exist_ok=True)

YAHOO = {
    "gold":   ["GC=F"],
    "silver": ["SI=F"],
    "wheat":  ["ZW=F"],
    "gaz":    ["NG=F"],
    "sp500":  ["^GSPC"],
    "nsdq":   ["^IXIC"],
    "vix":    ["^VIX"],
    "ibex":   ["^IBEX"],
    "aex":    ["^AEX"],
    "bel":    ["^BFX"],
    "dax":    ["^GDAXI"],
    "ftse":   ["^FTSE"],
    "smi":    ["^SSMI"],
    "asx":    ["^AXJO"],
    "n225":   ["^N225"],
    "his":    ["^HSI"],
    "nifty":  ["^NSEI"],
    "bvsp":   ["^BVSP"],
    "nzx":    ["^NZ50"],
    "usyield": ["^TNX"],
    "ta35":   ["TA35.TA"],
    "oil": ["CL=F"]
}

In [ ]:
def fetch_single_asset(name):
    ticker = YAHOO[name]

    try:
        data = yf.download(
            tickers=ticker,
            interval=PERIOD,
            start=START,
            end=END
        )

        close = data["Close"].iloc[:, 0]
        
        close.name = name
        return close

    except Exception as e:
        print(f"{name}: {e}")
        return None

def fetch_all():

    # Fetch Yahoo data
    dico = {}

    for name in YAHOO.keys():
        result = fetch_single_asset(name)

        if result is not None:
            dico[name] = result

    yahoo_df = pd.concat(dico, axis=1, sort=True)

    # Fetch US yield data
    url = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=DGS10"
    r = requests.get(url, timeout=30)
    r.raise_for_status()

    df = pd.read_csv(
        io.StringIO(r.text),
        index_col=0,
        parse_dates=True
    )

    usyield = pd.to_numeric(df.iloc[:, 0], errors="coerce").loc[START:END]
    usyield.name = "us_yield"

    # Fetch Japan 10Y yield
    url = "https://www.mof.go.jp/english/policy/jgbs/reference/interest_rate/historical/jgbcme_all.csv"
    r = requests.get(url, timeout=30)
    r.raise_for_status()

    df = pd.read_csv(url, skiprows=1, na_values=["-"])
    df["Date"] = pd.to_datetime(df["Date"])

    jpyield = pd.to_numeric(
        df.set_index("Date")["10Y"],
        errors="coerce"
    ).loc[START:END]

    jpyield.name = "jp_10yield"

    # Combine all data
    data = pd.concat(
        [yahoo_df, usyield, jpyield],
        axis=1,
        sort=True
    )

    return data

def check_data_validity(data: pd.DataFrame) -> pd.DataFrame:

    # Check the validity of the timestamps
    if not isinstance(data.index, pd.DatetimeIndex):
        print("The index of the data is not a DatetimeIndex.")
        data.index = pd.to_datetime(data.index, errors="raise")
        print("Solved the problem. The indexes are now datetimes.\n")

    # Check the ordering of observations
    if not data.index.is_monotonic_increasing:
        print("The observations are not properly ordered.")
        data.sort_index(ascending=True, inplace=True)
        print("The observations are now properly ordered.\n")

    # Check for duplicate timestamps
    if not data.index.is_unique:
        print("The index contains duplicate timestamps.")
        data = data.loc[~data.index.duplicated(keep="last")]
        print("Duplicate timestamps removed; last observation kept.\n")

    # Check for missing values
    missing_cols = data.columns[data.isna().any()]

    if len(missing_cols) > 0:
        print("Missing values detected.")
        data.loc[:, missing_cols] = data.loc[:, missing_cols].ffill()

        remaining_missing = data.columns[data.isna().any()]

        if len(remaining_missing) > 0:
            print("Some missing values remain after forward filling. Applying backward filling for leading missing values.")
            data.loc[:, remaining_missing] = (data.loc[:, remaining_missing].bfill())

        print("Missing-value handling completed.\n")

    print("Data validity check completed successfully.")
    
    return data

def build_features(data: pd.DataFrame, lag_order: int = 10, target_col: str = "oil") -> tuple[pd.DataFrame]:

    data = data.copy()

    return_cols = data.columns.tolist()
    predictions_cols = []

    # transform the VIX in a dummy variable
    data["vix_dummy"] = (data["vix"] > 20).astype("int8")
    predictions_cols.append("vix_dummy")

    # moving_averages on the target_col up order 10. we exclude order 1 since it is equivalent to just seeing lag 1. It would introduce a duplicate column
    ma_features = {
        f"{target_col}_ma{i}": data[target_col].rolling(window=i, min_periods=2).mean()
        for i in range(2, lag_order + 1)
    }
    predictions_cols.extend(ma_features.keys())
    data = pd.concat([data, pd.DataFrame(ma_features, index=data.index)],axis=1)

    # transform every columns into returns except the vix column
    # I did a forward:backward fill so when computing returns, there will be articial zeros in the dataset. We should think about it
    data[return_cols] = data[return_cols].pct_change(fill_method=None)

    # compute lag up to order 10 of all the columns
    lag_features = {
        f"{col}_lag_{i}": data[col].shift(i)
        for col in return_cols
        for i in range(1, lag_order+1)
    }
    predictions_cols.extend(lag_features.keys())
    data = pd.concat([data, pd.DataFrame(lag_features, index=data.index)], axis=1)

    # add the target_col to prediction_cols to complete the prediction dataframe
    predictions_cols.append(target_col)

    return data.dropna()[predictions_cols], data[return_cols]

def visualize_returns(returns_data: pd.DataFrame) -> None:
    for col in returns_data.columns:
        fig, ax = plt.subplots(figsize=(12, 4))

        ax.plot(returns_data.index, returns_data[col], linewidth=0.8)
        ax.set_title(f"Returns of {col}")
        ax.set_xlabel("Date")
        ax.set_ylabel("Return")
        ax.grid(True, alpha=0.3)

        plt.tight_layout()
        plt.show()
        

In [ ]:
allData = fetch_all()
cleaned_data = check_data_validity(allData)
prediction_dataset, returns_data = build_features(cleaned_data)